In [1]:
import torch
import torch.nn as nn

In [2]:
torch.manual_seed(42)

ratings = torch.tensor([
    [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 1, 1],
], dtype=torch.float32)

In [3]:
num_users, num_items = ratings.shape

users = []
items = []
labels = []

for user_id in range(num_users):
  for item_id in range(num_items):
    # 사용자 id들 전부 추가
    users.append(user_id)
    # 아이템 종류 클래스 추가
    items.append(item_id)
    # np.flatten(ratings)
    labels.append(ratings[user_id, item_id])

users = torch.tensor(users)
items = torch.tensor(items)
labels = torch.tensor(labels)

In [6]:
users.size(), items.size(), labels.size()

(torch.Size([50]), torch.Size([50]), torch.Size([50]))

In [ ]:
class Recommender(nn.Module):
  def __init__(self, num_users, num_items, embedding_dim=4):
    super().__init__()

    # 사용자 임베딩한 형태 (5명에 대해서 Size[4] 짜리 벡터로 변환)
    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    # 아이템 임베딩한 형태 (10개의 아이템에 대해서 모두 Size[4]짜리 벡터로 변환)
    self.item_embedding = nn.Embedding(num_items, embedding_dim)

  def forward(self, user_ids, item_ids):
    user_vec = self.user_embedding(user_ids)
    item_vec = self.item_embedding(item_ids)

    # 사용자와 아이템 임베딩을 내적하여 코사인 유사도를 구하기
    # user_vec: [n, 4]
    # item_vec: [n, 4]
    # 결과로 (n,)가 나옴 (벡터 유사도)
    score = (user_vec * item_vec).sum(dim=1)
    return score

In [ ]:
model = Recommender(
  num_users=num_users,
  num_items=num_items,
  embedding_dim=4
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(500):
  optimizer.zero_grad()

  # users: Size[50]
  # items: Size[50]
  # scores: Size[50]
  scores = model(users, items)

  # loss 수식 + 값
  # 현재 criterion은 BCEWithLoss 로 이진 분류에서 사용되는 손실함수
  # logits를 sigmoid에 넣어서 사용해줌.
  loss = criterion(scores, labels)

  loss.backward()
  optimizer.step()

  if epoch % 100 == 0:
    print(epoch, loss.item())

0 1.0266563892364502
100 0.0037978366017341614
200 0.0010861026821658015
300 0.0005418378277681768
400 0.0003288783482275903


In [12]:
user_id = 0


user_tensor = torch.full(
  (num_items,),
  user_id,
  dtype=torch.long
)
item_tensor = torch.arange(num_items)

user_tensor, item_tensor

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]))

In [17]:


with torch.no_grad():
  # 새로 만든 값을 맞춰보기
  scores = model(user_tensor, item_tensor)
  # 각각 벡터 ㅇ사도 구하기
  probabilities = torch.sigmoid(scores)

probabilities

tensor([9.9955e-01, 9.9973e-01, 5.8577e-04, 1.0000e+00, 1.5491e-04, 1.9593e-11,
        2.4285e-08, 5.7305e-04, 2.4168e-04, 3.8152e-04])

In [ ]:
# 순위중에서 높은 3개 뽑아보기
top_scores, top_items = torch.topk(probabilities, k=3)

# 아이템 클래스 3번을 제일 좋아하고, 1번, 0번을 그 다음으로 좋아할 확률이 높다는 뜻
print(top_items)
print(top_scores)

tensor([3, 1, 0])
tensor([1.0000, 0.9997, 0.9996])


In [20]:
# (사용자, 아이템)에 대한 선택을 float32 타입으로 만들어주기
ratings = torch.tensor([
    [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 1, 1],
], dtype=torch.float32)

In [26]:
num_users, num_items = ratings.shape

test_items = {}

for user_id in range(num_users):
  # 사용자 선택이 1일 인덱스들을 모두 뽑아보기
  # where 결과는 튜플임
  positive_items = torch.where(ratings[user_id] == 1)[0]
  print(f"{user_id}: {positive_items}")

  # test_items의 사용자별 선택에서 마지막 아이템만 추가해주기
  test_items[user_id] = positive_items[-1].item()

print(test_items)

0: tensor([0, 1, 3])
1: tensor([2, 3, 4])
2: tensor([4, 5, 6])
3: tensor([0, 6, 7])
4: tensor([1, 7, 8, 9])
{0: 3, 1: 4, 2: 6, 3: 7, 4: 9}


In [36]:
users = []
items = []
labels = []

for user_id in range(num_users):
  for item_id in range(num_items):
    # user_id, item_id 로 순회

    # 사용자의 마지막 아이템 선택이면 제외 (얼마나 맞는지 예측하기 위한 검증값으로 사용할 예정)
    if item_id == test_items[user_id]:
      continue

    users.append(user_id)
    items.append(item_id)
    # 각각의 선택에 대한 0, 1 값 저장해주기
    labels.append(ratings[user_id, item_id])

users = torch.tensor(users)
items = torch.tensor(items)
labels = torch.tensor(labels)

users.shape, items.shape, labels.shape

(torch.Size([45]), torch.Size([45]), torch.Size([45]))

In [29]:
class Recommender(nn.Module):
  def __init__(self, num_users, num_items, embedding_dim=4):
    super().__init__()

    # 사용자 id별 임베딩 벡터 만들어주기
    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    # 아이템 유형별 벡터들 만들어주기
    self.item_embedding = nn.Embedding(num_items, embedding_dim)

  def forward(self, user_ids, item_ids):
    # 사용자 번호에 대해서 vector로 변환해주기
    user_vec = self.user_embedding(user_ids)
    # 아이템 번호에 대해서 vector로 변환해주기
    item_vec = self.item_embedding(item_ids)

    return (user_vec * item_vec).sum(dim=1)

In [34]:
model = Recommender(num_users, num_items, embedding_dim=4)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(500):
  optimizer.zero_grad()

  # 사용자 ids와 item_ids를 이용해서 벡터 유사도들 구해서 받아주기
  scores = model(users, items)
  # 손실함수 계산해주기
  loss = criterion(scores, labels)

  # 손실함수에 대해서 역저파 -> 기울기 누적 시켜주기
  loss.backward()

  optimizer.step()

In [39]:
user_id = 0
# 사용자 텐서를 아이템 길이 (10, )만큼 만들어주기. 0으로 채워진 길이 10, dtype long 짜리 tensor이 만들어지게 됨
user_tensor  = torch.full(
  (num_items,),
  user_id,
  dtype=torch.long
)

# 각각의 아이템에 대해서 사용자의 선호들을 뽑기 위해서 클래스 배열을 뽑게 됨
item_tensor = torch.arange(num_items)


with torch.no_grad():
  # 사용자 0에 대한 모든 아이템들의 예측을 해보기
  scores = model(user_tensor, item_tensor)
  print(scores)

# 결과상 0번을 가장 좋아하며 그 다음이 1, 3 순서임

tensor([ 10.4277,   7.9138,  -9.4531,   1.2732, -12.2271, -16.5489,  -8.1458,
         -8.7261,  -8.4681,  -9.5162])


In [45]:
# 사용자의 선택한 ratings의 인덱스들을 모두 뽑음
seen_items = torch.where(ratings[user_id]==1)[0].tolist()

# 사용자 id에 대한 아이템 선택값들을 뽑음
test_item = test_items[user_id]

# 이미 봤던 내용 중에서 test_item을 미리 빼놓음
seen_items.remove(test_item)

print(seen_items)

scores[seen_items] = -float("inf")

[0, 1]


In [46]:
top_scores, top_items = torch.topk(scores, k=3)

print(top_items)

tensor([3, 6, 8])


In [49]:
import math

# 
def ndcg_at_k(top_items, target_item, k):
  """top_k 에서 target_item이 top_items의 몇번째에 있는지 확인"""
  # 상위 n개를 리스트로 뽑기
  top_items = top_items[:k].tolist()

  # 원하는 아이템이 없으면 0점 반환
  if target_item not in top_items:
    return 0.0

  # 몇등으로 있는지 찾기
  rank = top_items.index(target_item) + 1

  return 1 / math.log2(rank+1)

In [50]:
score = ndcg_at_k(top_items, test_items[user_id], k=3)
score

1.0